In [ ]:
import pandas as pd
import numpy as np
import os

W_SIGLIP = 0.4
W_DINO   = 0.6

FILES = {
    'siglip': 'submission70.csv',
    'dino':   'submission72.csv'
}

OUTPUT_FILE = 'submission.csv'

ALL_TARGETS = ['Dry_Green_g', 'Dry_Clover_g', 'Dry_Dead_g', 'GDM_g', 'Dry_Total_g']


def enforce_mass_balance(df_wide, fixed_clover=None):
    """
    Applies Orthogonal Projection to enforce biological constraints:
    1. Dry_Green_g + Dry_Clover_g = GDM_g
    2. GDM_g + Dry_Dead_g = Dry_Total_g
    
    If fixed_clover is True, Dry_Clover_g is kept fixed and only other targets are adjusted.
    
    This finds the closest set of values to the predictions that satisfy 
    the constraints (minimizing Euclidean distance modification).
    """
    ordered_cols = ['Dry_Green_g', 'Dry_Clover_g', 'Dry_Dead_g', 'GDM_g', 'Dry_Total_g']
    
    Y = df_wide[ordered_cols].values.T
    
    if fixed_clover:
        clover_fixed = Y[1, :].copy()  # Dry_Clover_g is index 1
        Y[3, :] = Y[0, :] + clover_fixed  # GDM_g
        Y[4, :] = Y[3, :] + Y[2, :]  # Dry_Total_g
        Y_reconciled = Y
    else:
        C = np.array([
            [1, 1, 0, -1,  0],
            [0, 0, 1,  1, -1]
        ])
        
        C_T = C.T
        try:
            inv_CCt = np.linalg.inv(C @ C_T)
            P = np.eye(5) - C_T @ inv_CCt @ C
        except np.linalg.LinAlgError:
            print("Warning: Singular matrix in projection. Skipping constraint enforcement.")
            return df_wide

        Y_reconciled = P @ Y
    
    Y_reconciled = Y_reconciled.T
    Y_reconciled = np.maximum(0, Y_reconciled) 
    
    df_out = df_wide.copy()
    df_out[ordered_cols] = Y_reconciled
    
    return df_out

def robust_ensemble(file_paths, weights):
    print(f"--- Starting Ensemble ---")
    print(f"Weights: {weights}")
    print("NOTE: Using DINO-only for Dry_Clover_g (better detection)")
    
    dfs = []
    for name, path in file_paths.items():
        if not os.path.exists(path):
            raise FileNotFoundError(f"Missing file: {path}")
        
        df = pd.read_csv(path).sort_values('sample_id').reset_index(drop=True)
        dfs.append(df)
        print(f"Loaded {name}: {len(df)} rows")

    base_ids = dfs[0]['sample_id']
    if not all(df['sample_id'].equals(base_ids) for df in dfs[1:]):
        raise ValueError("Sample IDs do not match between submission files!")

    dfs_split = []
    for df in dfs:
        df_split = df.copy()
        df_split[['image_id', 'target_name']] = df_split['sample_id'].str.rsplit('__', n=1, expand=True)
        dfs_split.append(df_split)
    
    dino_idx = list(file_paths.keys()).index('dino')
    siglip_idx = list(file_paths.keys()).index('siglip')
    dino_df = dfs_split[dino_idx]
    siglip_df = dfs_split[siglip_idx]
    
    ensemble_results = []
    
    for target in ALL_TARGETS:
        dino_target = dino_df[dino_df['target_name'] == target].copy()
        siglip_target = siglip_df[siglip_df['target_name'] == target].copy()
        
        if target == 'Dry_Clover_g':
            print(f"Using DINO-only for {target}")
            ensemble_results.append(dino_target[['sample_id', 'target']])
        else:
            merged = pd.merge(
                dino_target[['sample_id', 'target']].rename(columns={'target': 'dino_pred'}),
                siglip_target[['sample_id', 'target']].rename(columns={'target': 'siglip_pred'}),
                on='sample_id'
            )
            
            w_vec = np.array([weights['dino'], weights['siglip']])
            w_vec = w_vec / w_vec.sum()
            
            merged['target'] = merged['dino_pred'] * w_vec[0] + merged['siglip_pred'] * w_vec[1]
            ensemble_results.append(merged[['sample_id', 'target']])
    
    ensemble_df = pd.concat(ensemble_results, ignore_index=True)
    print("Weighted average complete (DINO-only for Clover).")

    ensemble_df[['image_id', 'target_name']] = ensemble_df['sample_id'].str.rsplit('__', n=1, expand=True)
    
    wide_df = ensemble_df.pivot(index='image_id', columns='target_name', values='target').reset_index()

    print("Applying Mass Balance Constraints (Dry_Clover_g fixed to DINO values)...")
    wide_balanced = enforce_mass_balance(wide_df, fixed_clover=True)

    long_balanced = wide_balanced.melt(
        id_vars='image_id', 
        value_vars=ALL_TARGETS,
        var_name='target_name',
        value_name='target'
    )

    long_balanced['sample_id'] = long_balanced['image_id'] + '__' + long_balanced['target_name']

    final_submission = long_balanced[['sample_id', 'target']].sort_values('sample_id').reset_index(drop=True)

    return final_submission

if __name__ == "__main__":
    
    ensemble_weights = {
        'siglip': W_SIGLIP,
        'dino': W_DINO
    }
    
    try:
        submission = robust_ensemble(FILES, ensemble_weights)
        
        submission.to_csv(OUTPUT_FILE, index=False)
        print(f"\nSuccess! Saved to {OUTPUT_FILE}")
        print(submission.head())
        
        print("\nStats:")
        print(submission['target'].describe())
        
    except Exception as e:
        print(f"\nError during ensembling: {e}")